In [ ]:
import subprocess
import os
import sys
import torch
import torch.nn.functional as F
import numpy as np
from ood_metrics import fpr_at_95_tpr, calc_metrics, plot_roc, plot_pr,plot_barcode
from sklearn.metrics import roc_auc_score, roc_curve, auc, precision_recall_curve, average_precision_score
from torchvision.transforms import Compose, Resize, ToTensor, Normalize

eval_path = os.path.abspath("eval")
if eval_path not in sys.path:
    sys.path.insert(0, eval_path)
from pathGTComparison import maskGt

In [ ]:
datasets = [
    "FS_LostFound_full",
    "RoadObsticle21",
    "fs_static",
    "RoadAnomaly",
    "RoadAnomaly21"
]

weight_conf_mapping_eomt = {
    "eomt/weights/eomt_cityscapes.bin": [
        "eomt/configs/dinov2/cityscapes/semantic/eomt_base_640.yaml"
    ],
    "eomt/weights/eomt_coco.bin": [
        "eomt/configs/dinov2/coco/panoptic/eomt_base_640_2x.yaml"
    ],
}



### Anomaly evaluation for Erfnet

In [ ]:
for ds_name in datasets:
                
    input_glob = f"datasets/Validation_Dataset/{ds_name}/images/*.*"
    
    command = [
        "python", "eval/evalAnomaly.py",
        "--input", input_glob,
        "--loadDataset", ds_name,
        "--loadDir", "trained_models/",
        "--loadWeights", "erfnet_pretrained.pth",
        "--batch-size", "1"
    ]
    
    print(f"\n{'='*42}")
    print(f"RUNNING: DS={ds_name} | WEIGHTS=ERFNET_PRETRAINED")
    print(f"{'='*42}\n")
    
    try:
        subprocess.run(command, check=True)
    except subprocess.CalledProcessError as e:
        print(f"{ds_name} failed: {e}")
        continue

### Anomaly evaluation for Eomt

In [ ]:
for weight_path, configs in weight_conf_mapping_eomt.items():
        for conf_path in configs:
            for ds_name in datasets:
                
                input_glob = f"datasets/Validation_Dataset/{ds_name}/images/*.*"
                
                command = [
                    "python", "eval/evalAnomalyForEomt.py",
                    "--input", input_glob,
                    "--loadDataset", ds_name,
                    "--loadWeights", weight_path,
                    "--loadConf", conf_path
                ]
                
                print(f"\n{'='*42}")
                print(f"RUNNING: DS={ds_name} | WEIGHTS={os.path.basename(weight_path)}")
                print(f"CONF: {os.path.basename(conf_path)}")
                print(f"{'='*42}\n")
                
                try:
                    subprocess.run(command, check=True)
                except subprocess.CalledProcessError as e:
                    print(f"{ds_name} failed: {e}")
                    continue

### Temperature Scaling

In [ ]:
import os
import torch
import numpy as np
import torch.nn.functional as F
from PIL import Image
 

IMG_SIZE = (1024, 1024)
target_transform = Compose(
    [
        Resize(IMG_SIZE, Image.NEAREST),
    ]
)


temperatures = [0.5, 0.75, 1.1]

os.makedirs("results_anomaly", exist_ok=True)
csv_file_path = 'results_anomaly/results_temperature_scaling.csv'

if not os.path.exists(csv_file_path):
    with open(csv_file_path, 'w') as f:
        f.write("Model,Dataset,Temperature,AUPRC,FPR@TPR95\n")

for weight_path, _ in weight_conf_mapping_eomt.items(): 
    for ds_name in datasets: 
        model_name = weight_path.split('/')[-1].split('.')[0]
        logits_path = f"logits/{ds_name}/{model_name}"
        
        logits_files = sorted([
            os.path.join(logits_path, f) 
            for f in os.listdir(logits_path)
            if f.endswith(".pt")
        ])
        
        msp_lists = {t: [] for t in temperatures}
        ood_gts_list = []

        for l in logits_files: 
            logit = torch.load(l)
            
            for t in temperatures:
                l_scaled = logit / t
                prob = F.softmax(l_scaled, dim=0)
                msp = 1.0 - np.max(prob.squeeze(0).cpu().numpy(), axis=0)
                msp_lists[t].append(msp)
            

            base_name = os.path.basename(l).replace("logits_", "").replace(".pt", "")
            

            pathGT = f"datasets/Validation_Dataset/{ds_name}/labels_masks/{base_name}.png" 

            ood_gts = maskGt(pathGT, target_transform)
            if ood_gts is None: 
                continue
            
            ood_gts_list.append(ood_gts)


        ood_gts_array = np.array(ood_gts_list)
        ood_mask = (ood_gts_array == 1)
        ind_mask = (ood_gts_array == 0) 

        with open(csv_file_path, 'a') as file:
            for t in temperatures:
                scores = np.array(msp_lists[t])
                
                ood_out = scores[ood_mask]
                ind_out = scores[ind_mask]

                val_out = np.concatenate((ind_out, ood_out))
                val_label = np.concatenate((np.zeros(len(ind_out)), np.ones(len(ood_out))))

                if len(np.unique(val_label)) > 1:
                    prc_auc = average_precision_score(val_label, val_out)
                    fpr = fpr_at_95_tpr(val_out, val_label)
                    
                    file.write(f"{model_name},{ds_name},Temp_{t},{prc_auc*100.0:.2f},{fpr*100.0:.2f}\n")
                    print(f"[{model_name} - {ds_name} - T={t}] AUPRC: {prc_auc*100:.2f}% | FPR95: {fpr*100:.2f}%")
                else:
                    print(f"[{model_name} - {ds_name} - T={t}] No anomaly found")

